# Dataset Verification
Extracted patch들의 품질과 분포를 시각적으로 검증

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import random
import openslide

random.seed(42)

PATCH_DIR = Path('/app/Gland_Seg/patches')
metadata = pd.read_csv(PATCH_DIR / 'metadata.csv')
print(f'Total patches: {len(metadata)}')
print(f'\nPer slide:')
print(metadata.groupby(['slide', 'class']).size().to_string())
print(f'\nClass distribution:')
print(metadata['class'].value_counts().to_string())

In [ ]:
# Random sample grid: 10 per class
fig, axes = plt.subplots(2, 10, figsize=(25, 6))

for row_idx, cls in enumerate(['gland', 'non-gland']):
    cls_df = metadata[metadata['class'] == cls]
    samples = cls_df.sample(10, random_state=42)
    for col_idx, (_, row) in enumerate(samples.iterrows()):
        img_path = PATCH_DIR / row['slide'] / row['class'] / row['filename']
        img = Image.open(img_path)
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx, col_idx].set_ylabel(cls.upper(), fontsize=14, fontweight='bold')

axes[0, 4].set_title('Gland forming cancer', fontsize=14, fontweight='bold')
axes[1, 4].set_title('Non-gland cancer (Solid)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/app/Gland_Seg/Viz/sample_patches.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Patch locations overlay on slide thumbnails
from config import Config
config = Config()

n_slides = len(config.slides)
fig, axes = plt.subplots(1, n_slides, figsize=(6 * n_slides, 10))

for idx, (slide_name, info) in enumerate(config.slides.items()):
    svs_path = Path(config.svs_dir) / info['svs']
    slide = openslide.OpenSlide(str(svs_path))
    viz_level = min(3, slide.level_count - 1)
    ds = slide.level_downsamples[viz_level]
    thumb = slide.read_region((0, 0), viz_level, slide.level_dimensions[viz_level]).convert('RGB')
    thumb_np = np.array(thumb)

    axes[idx].imshow(thumb_np)

    slide_df = metadata[metadata['slide'] == slide_name]
    color = 'green' if info['class'] == 'gland' else 'orange'
    ps = config.patch_size / ds
    for _, row in slide_df.iterrows():
        rect = plt.Rectangle((row['x']/ds, row['y']/ds), ps, ps,
                           linewidth=0.3, edgecolor=color, facecolor=color, alpha=0.2)
        axes[idx].add_patch(rect)

    axes[idx].set_title(f"{slide_name}\n{info['class']} ({len(slide_df)} patches)", fontsize=11)
    axes[idx].axis('off')
    slide.close()

plt.tight_layout()
plt.savefig('/app/Gland_Seg/Viz/patch_locations.png', dpi=150, bbox_inches='tight')
plt.show()